# 04 v4 — Ekstraksi Pairwise `DINOv3 SAT-493M` + `phys_v4.parquet` + Offline `weights/` & `Tes A`

Notebook Tugas A (**Hafizh — Pengolah Gambar**) sesuai `pembagian_tugas_tim.md` & `perubahan_preprocessing_v4.md`:
1. **Backbone `DINOv3 SAT-493M` (`facebook/dinov3-vitl16-pretrain-sat493m`)** + otomatis menyimpan bobot ke folder lokal **`weights/`** agar `02_submission.ipynb` Seno dapat berjalan **100% offline tanpa internet/HF_TOKEN**.
2. **Preprocessing Terkunci 1:1 dengan `dino_v4_frozen_for_seno.py`:** pure asymmetric `pink_cast`, guarded shared valid-scale crop, post-norm zero masking, cloud-excluded ViT pooling (`cloud <= 0.50`), TTA 8 arah D4, `USE_FP16 = True`, `cudnn.deterministic = True`.
3. **Output Versi Baru (`v4_dino_features/` — tidak menimpa `v3_dino_features/`):** `dino_color_pre/post.npy`, `dino_gray_pre/post.npy`, `dino_changed_embed.npy`, `artifact_and_phys_v3.parquet`, `patch_change_stats_v3.parquet`, `phys_v4.parquet` (`23` kolom), `extract_v4_meta.json` (`meta.json`), dan `requirements_extract.txt`.
4. **Verifikasi Otomatis:** Evaluasi **Tes A** (`Recall@1`, `Recall@5`, `Recall@10` warna vs grayscale) + **Audit Bagian 5 `phys_v4`** (`|corr| < 0.30`, `% color_unreliable_v4`, probe ID).

In [1]:
!pip install -q -U "transformers>=4.56.0" pyarrow accelerate

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [2]:
import gc
import os
import json
import time
import random
import warnings
warnings.filterwarnings("ignore")

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from scipy.ndimage import distance_transform_edt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
try:
    import timm
except ImportError:
    timm = None
try:
    import transformers
except ImportError:
    transformers = None


# ==================== KONFIGURASI TERKUNCI (V4 CONTRACT) ====================
SEED = 42
BACKBONE_CHOICE = "dinov3_sat_large"   # Pilihan: "dinov3_sat_large", "dinov2_large", "dinov2_base"
IMG_SIZE_V3 = 256                      # Kelipatan 16 -> 16x16 = 256 patch untuk DINOv3 SAT-493M
DINOV2_FALLBACK_IMG_SIZE = 308         # Kelipatan 14 -> 22x22 = 484 patch untuk DINOv2-L / DINOv2-B
BATCH_SIZE = 16
NUM_WORKERS = 2
EXTRACT_GRAY3 = True
USE_FP16 = True                        # Sesuai keputusan user: FP16 autocast (sama persis di extractor & submission)
ALLOW_DATAPARALLEL = False             # Default 1 GPU agar konsisten dengan 1x Kaggle T4 di 02_submission.ipynb
TINT_NORM_THR = 0.05                   # Pure asymmetric max(0, min(R-G, B-G))
SHARED_CROP_MIN_RATIO = 0.50           # Side shared jika min/max >= 0.50
CLOUD_PATCH_THR = 0.50                 # Maksimal fraksi awan per patch untuk pooling post/changed
# ===========================================================================


def set_reproducible_seed(seed: int = SEED):
    """Kunci RNG dan flag deterministik PyTorch/cuDNN agar rerun konsisten."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_hf_token():
    tok = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
    if tok:
        return tok.strip()
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF_TOKEN")
        if tok:
            return tok.strip()
    except Exception:
        pass
    try:
        from google.colab import userdata
        tok = userdata.get("HF_TOKEN")
        if tok:
            return tok.strip()
    except Exception:
        pass
    return None


def detect_valid_mask(rgb: np.ndarray, sum_thr: int = 10) -> np.ndarray:
    """Deteksi area piksel non-hitam valid."""
    return rgb.sum(axis=2) > sum_thr


def _extract_valid_crop(rgb: np.ndarray):
    valid_raw = detect_valid_mask(rgb)
    if int(valid_raw.sum()) < 50:
        return rgb.copy(), np.ones(rgb.shape[:2], dtype=bool), max(rgb.shape[:2]), False
    ys, xs = np.where(valid_raw)
    y0, y1 = max(0, int(ys.min()) - 4), min(rgb.shape[0], int(ys.max()) + 5)
    x0, x1 = max(0, int(xs.min()) - 4), min(rgb.shape[1], int(xs.max()) + 5)
    crop = rgb[y0:y1, x0:x1].copy()
    m_crop = valid_raw[y0:y1, x0:x1]
    side = max(crop.shape[0], crop.shape[1])
    return crop, m_crop, side, True


def _pad_inpaint_resize(crop: np.ndarray, m_crop: np.ndarray, side: int, out_size: int):
    h, w = crop.shape[:2]
    side = max(side, h, w)
    py, px = (side - h) // 2, (side - w) // 2
    sq_img = np.zeros((side, side, 3), dtype=np.uint8)
    sq_msk = np.zeros((side, side), dtype=bool)
    sq_img[py:py + h, px:px + w] = crop
    sq_msk[py:py + h, px:px + w] = m_crop

    if not sq_msk.all() and sq_msk.any():
        _, idx = distance_transform_edt(~sq_msk, return_indices=True)
        filled = sq_img.copy()
        filled[~sq_msk] = sq_img[idx[0][~sq_msk], idx[1][~sq_msk]]
        blurred = cv2.GaussianBlur(filled, (15, 15), 0)
        sq_img[~sq_msk] = blurred[~sq_msk]

    interp = cv2.INTER_AREA if side >= out_size else cv2.INTER_LINEAR
    return (
        cv2.resize(sq_img, (out_size, out_size), interpolation=interp),
        cv2.resize(sq_msk.astype(np.float32), (out_size, out_size), interpolation=cv2.INTER_AREA),
    )


def crop_pad_inpaint(rgb: np.ndarray, out_size: int, force_side: int = None):
    """Crop ketat bounding box valid, pad persegi (opsional shared side), dan inpaint tepi kosong."""
    crop, m_crop, side, ok = _extract_valid_crop(rgb)
    if not ok:
        return (
            cv2.resize(crop, (out_size, out_size), interpolation=cv2.INTER_AREA),
            np.ones((out_size, out_size), dtype=np.float32),
        )
    use_side = side if force_side is None else max(force_side, side)
    return _pad_inpaint_resize(crop, m_crop, use_side, out_size)


def crop_pad_inpaint_pair(rgb_pre: np.ndarray, rgb_post: np.ndarray, out_size: int):
    """Gunakan side_shared jika rasio skala pre/post >= 0.50 agar resolusi spasial seragam."""
    cp, mp, sp, ok_p = _extract_valid_crop(rgb_pre)
    cq, mq, sq, ok_q = _extract_valid_crop(rgb_post)
    if not ok_p or not ok_q:
        return crop_pad_inpaint(rgb_pre, out_size), crop_pad_inpaint(rgb_post, out_size), False

    ratio = min(sp, sq) / float(max(sp, sq) + 1e-6)
    if ratio >= SHARED_CROP_MIN_RATIO:
        side_shared = min(1024, max(sp, sq))
        return _pad_inpaint_resize(cp, mp, side_shared, out_size), _pad_inpaint_resize(cq, mq, side_shared, out_size), True
    return _pad_inpaint_resize(cp, mp, sp, out_size), _pad_inpaint_resize(cq, mq, sq, out_size), False


def selective_color_norm(rgb_sq: np.ndarray, msk_sq: np.ndarray, tint_thr: float = TINT_NORM_THR) -> np.ndarray:
    """Detektor tint asimetris murni (R > G AND B > G) pada skala [0, 255]; pertahankan hutan/laut alami."""
    arr = rgb_sq.astype(np.float32)
    v = msk_sq > 0.5
    if int(v.sum()) < 30:
        v = np.ones_like(msk_sq, dtype=bool)
    rm, gm, bm = arr[:, :, 0][v].mean(), arr[:, :, 1][v].mean(), arr[:, :, 2][v].mean()
    tint = max(0.0, min(float(rm - gm), float(bm - gm))) / 255.0
    if tint > tint_thr:
        mu = np.array([rm, gm, bm], dtype=np.float32)[None, None, :]
        sd = np.maximum(np.array([arr[:, :, c][v].std() for c in range(3)], dtype=np.float32), 12.0)[None, None, :]
        return np.clip(127.5 + 48.0 * ((arr - mu) / sd), 0.0, 255.0).astype(np.uint8)
    return rgb_sq


def make_gray3_clahe(rgb_sq: np.ndarray) -> np.ndarray:
    """Cabang grayscale CLAHE 3-channel yang kebal acak warna/tint."""
    g = cv2.cvtColor(rgb_sq, cv2.COLOR_RGB2GRAY)
    cg = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(g)
    return np.stack([cg, cg, cg], axis=-1)


class PatchTokenWrapper(nn.Module):
    """Ekstrak patch spasial murni dengan membuang prefix token (CLS + register tokens) secara dinamis."""
    def __init__(self, core_model, is_hf: bool, n_prefix: int):
        super().__init__()
        self.core_model = core_model
        self.is_hf = is_hf
        self.n_prefix = n_prefix

    def forward(self, x):
        if self.is_hf:
            tokens = self.core_model(pixel_values=x).last_hidden_state
        else:
            tokens = self.core_model.forward_features(x)
        return tokens[:, self.n_prefix:, :]


def _has_hf_weights(d: str) -> bool:
    return bool(d) and os.path.isdir(d) and os.path.exists(os.path.join(d, "config.json")) and (
        os.path.exists(os.path.join(d, "model.safetensors")) or os.path.exists(os.path.join(d, "pytorch_model.bin"))
    )


def _has_timm_weights(d: str) -> bool:
    return bool(d) and os.path.isdir(d) and os.path.exists(os.path.join(d, "model_state.pt")) and os.path.exists(os.path.join(d, "backbone_config.json"))


def find_local_weights_dir(backbone_choice: str = BACKBONE_CHOICE, explicit_dir: str = None):
    """Cari folder bobot lokal di ./weights, /kaggle/working/weights, atau /kaggle/input/ agar bisa jalan offline."""
    candidates = []
    if explicit_dir:
        candidates.append(explicit_dir)
    for base in ["./weights", "/kaggle/working/weights", "/kaggle/input"]:
        if not os.path.exists(base):
            continue
        candidates.append(os.path.join(base, backbone_choice))
        if base == "/kaggle/input":
            for root, dirs, _ in os.walk(base):
                if os.path.basename(root) == backbone_choice:
                    candidates.append(root)
                if "weights" in dirs:
                    candidates.append(os.path.join(root, "weights", backbone_choice))
    for cand in candidates:
        if _has_hf_weights(cand) or _has_timm_weights(cand):
            return cand
    return None


def _build_meta_info(base_info: dict, device: str, loaded_from: str):
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
    base_info.update({
        "seed": SEED,
        "tint_norm_thr": TINT_NORM_THR,
        "cloud_pool_thr": CLOUD_PATCH_THR,
        "shared_crop_min_ratio": SHARED_CROP_MIN_RATIO,
        "precision": "fp16_autocast" if (USE_FP16 and device == "cuda") else "fp32",
        "device": device,
        "gpu_name": gpu_name,
        "n_gpus_visible": int(torch.cuda.device_count()) if torch.cuda.is_available() else 0,
        "loaded_from": loaded_from,
        "torch_version": torch.__version__,
        "transformers_version": getattr(transformers, "__version__", "not_installed"),
        "timm_version": getattr(timm, "__version__", "not_installed"),
    })
    return base_info


def _save_backbone_meta(save_dir: str, info: dict):
    os.makedirs(save_dir, exist_ok=True)
    with open(os.path.join(save_dir, "backbone_config.json"), "w", encoding="utf-8") as f:
        json.dump(info, f, indent=2)


def load_dino_backbone(
    backbone_choice: str = BACKBONE_CHOICE,
    device: str = None,
    hf_token: str = None,
    local_weights_dir: str = None,
    save_weights_dir: str = None,
):
    """Muat backbone secara offline dari weights/ bila tersedia; jika online, download lalu simpan ke save_weights_dir."""
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    if hf_token is None:
        hf_token = get_hf_token()

    found_local = find_local_weights_dir(backbone_choice, explicit_dir=local_weights_dir)
    if found_local:
        if _has_hf_weights(found_local):
            from transformers import AutoImageProcessor, AutoModel
            proc = AutoImageProcessor.from_pretrained(found_local, local_files_only=True)
            hf_m = AutoModel.from_pretrained(found_local, local_files_only=True).eval().to(device)
            n_pref = 1 + int(getattr(hf_m.config, "num_register_tokens", 4 if "dinov3" in backbone_choice else 0))
            p_sz = int(getattr(hf_m.config, "patch_size", 16 if "dinov3" in backbone_choice else 14))
            img_sz = IMG_SIZE_V3 if "dinov3" in backbone_choice else DINOV2_FALLBACK_IMG_SIZE
            cfg_p = os.path.join(found_local, "backbone_config.json")
            saved_cfg = json.load(open(cfg_p, "r", encoding="utf-8")) if os.path.exists(cfg_p) else {}
            info = _build_meta_info({
                "model": saved_cfg.get("model", found_local),
                "backbone_choice": backbone_choice,
                "patch_size": int(saved_cfg.get("patch_size", p_sz)),
                "n_prefix": int(saved_cfg.get("n_prefix", n_pref)),
                "img_size": int(saved_cfg.get("img_size", img_sz)),
                "dim": int(hf_m.config.hidden_size),
                "mean": list(getattr(proc, "image_mean", saved_cfg.get("mean", [0.430, 0.411, 0.296]))),
                "std": list(getattr(proc, "image_std", saved_cfg.get("std", [0.213, 0.156, 0.143]))),
                "license": saved_cfg.get("license", "DINOv3 License" if "dinov3" in backbone_choice else "Apache-2.0"),
            }, device=device, loaded_from=f"offline:{found_local}")
            print(f"[OFFLINE OK] Model dimuat lokal dari {found_local}")
            return PatchTokenWrapper(hf_m, True, info["n_prefix"]), info

        if _has_timm_weights(found_local):
            saved_cfg = json.load(open(os.path.join(found_local, "backbone_config.json"), "r", encoding="utf-8"))
            m = timm.create_model(saved_cfg["model"], pretrained=False, num_classes=0, img_size=saved_cfg["img_size"]).eval()
            state = torch.load(os.path.join(found_local, "model_state.pt"), map_location="cpu")
            m.load_state_dict(state)
            m = m.to(device)
            info = _build_meta_info(saved_cfg, device=device, loaded_from=f"offline:{found_local}")
            print(f"[OFFLINE OK] Model timm dimuat lokal dari {found_local}")
            return PatchTokenWrapper(m, False, info["n_prefix"]), info

    if backbone_choice == "dinov3_sat_large":
        try:
            from transformers import AutoImageProcessor, AutoModel
            repo = "facebook/dinov3-vitl16-pretrain-sat493m"
            proc = AutoImageProcessor.from_pretrained(repo, token=hf_token)
            hf_m = AutoModel.from_pretrained(repo, token=hf_token).eval().to(device)
            n_pref = 1 + int(getattr(hf_m.config, "num_register_tokens", 4))
            p_sz = int(getattr(hf_m.config, "patch_size", 16))
            info = _build_meta_info({
                "model": repo,
                "backbone_choice": "dinov3_sat_large",
                "patch_size": p_sz,
                "n_prefix": n_pref,
                "img_size": IMG_SIZE_V3,
                "dim": int(hf_m.config.hidden_size),
                "mean": list(getattr(proc, "image_mean", [0.430, 0.411, 0.296])),
                "std": list(getattr(proc, "image_std", [0.213, 0.156, 0.143])),
                "license": "DINOv3 License",
            }, device=device, loaded_from=f"online:{repo}")
            if save_weights_dir:
                out_w = os.path.join(save_weights_dir, "dinov3_sat_large")
                os.makedirs(out_w, exist_ok=True)
                proc.save_pretrained(out_w)
                hf_m.save_pretrained(out_w)
                _save_backbone_meta(out_w, info)
                print(f"[SAVED WEIGHTS] Bobot DINOv3 disimpan untuk offline: {out_w}")
            return PatchTokenWrapper(hf_m, True, n_pref), info
        except Exception as e:
            print(f"[INFO] Fallback ke DINOv2 karena: {type(e).__name__}: {str(e)[:160]}")

    if backbone_choice == "dinov2_base":
        from transformers import AutoImageProcessor, AutoModel
        repo = "facebook/dinov2-base"
        proc = AutoImageProcessor.from_pretrained(repo)
        hf_m = AutoModel.from_pretrained(repo).eval().to(device)
        n_pref = 1 + int(getattr(hf_m.config, "num_register_tokens", 0))
        info = _build_meta_info({
            "model": repo,
            "backbone_choice": "dinov2_base",
            "patch_size": 14,
            "n_prefix": n_pref,
            "img_size": DINOV2_FALLBACK_IMG_SIZE,
            "dim": int(hf_m.config.hidden_size),
            "mean": list(getattr(proc, "image_mean", [0.485, 0.456, 0.406])),
            "std": list(getattr(proc, "image_std", [0.229, 0.224, 0.225])),
            "license": "Apache-2.0",
        }, device=device, loaded_from=f"online:{repo}")
        if save_weights_dir:
            out_w = os.path.join(save_weights_dir, "dinov2_base")
            os.makedirs(out_w, exist_ok=True)
            proc.save_pretrained(out_w)
            hf_m.save_pretrained(out_w)
            _save_backbone_meta(out_w, info)
            print(f"[SAVED WEIGHTS] Bobot DINOv2-Base disimpan untuk offline: {out_w}")
        return PatchTokenWrapper(hf_m, True, n_pref), info

    name = "vit_large_patch14_dinov2.lvd142m"
    m = timm.create_model(name, pretrained=True, num_classes=0, img_size=DINOV2_FALLBACK_IMG_SIZE).eval().to(device)
    n_pref = int(getattr(m, "num_prefix_tokens", 1))
    info = _build_meta_info({
        "model": name,
        "backbone_choice": "dinov2_large",
        "patch_size": 14,
        "n_prefix": n_pref,
        "img_size": DINOV2_FALLBACK_IMG_SIZE,
        "dim": int(m.num_features),
        "mean": [0.485, 0.456, 0.406],
        "std": [0.229, 0.224, 0.225],
        "license": "Apache-2.0",
    }, device=device, loaded_from=f"online:{name}")
    if save_weights_dir:
        out_w = os.path.join(save_weights_dir, "dinov2_large")
        os.makedirs(out_w, exist_ok=True)
        torch.save(m.state_dict(), os.path.join(out_w, "model_state.pt"))
        _save_backbone_meta(out_w, info)
        print(f"[SAVED WEIGHTS] Bobot DINOv2-Large disimpan untuk offline: {out_w}")
    return PatchTokenWrapper(m, False, n_pref), info


def d4_augment_batch(x: torch.Tensor, m: torch.Tensor, fl: bool, rot: int):
    """Terapkan salah satu dari 8 transformasi simetri D4 secara sinkron pada gambar & mask."""
    xf = torch.flip(x, [3]) if fl else x
    mf = torch.flip(m.float(), [2]) > 0.5 if fl else m
    xv = torch.rot90(xf, rot, [2, 3]) if rot > 0 else xf
    mv = torch.rot90(mf, rot, [1, 2]) if rot > 0 else mf
    return xv, mv


def extract_patch_grid(model, x_v: torch.Tensor, m_v: torch.Tensor, device: str = None, use_fp16: bool = USE_FP16):
    """Jalankan forward lalu kembalikan patch L2-normalized & pooled embedding ter-mask."""
    if device is None:
        device = str(x_v.device).split(":")[0]
    bs = x_v.shape[0]
    amp_enabled = bool(use_fp16 and device == "cuda")
    with torch.amp.autocast("cuda", enabled=amp_enabled, dtype=torch.float16):
        toks = model(x_v)
    toks_n = F.normalize(toks.float(), dim=2)
    mw = m_v.reshape(bs, -1, 1).float()
    pooled = F.normalize((toks_n * mw).sum(dim=1) / (mw.sum(dim=1) + 1e-6), dim=1)
    return toks_n, mw, pooled


def compute_spatial_stats(cmap: np.ndarray, vmask: np.ndarray, grid_sz: int):
    """Statistik spasial perubahan patch bebas rotasi berbasis P10 + P15 + floor absolut."""
    out = {}
    keys = (
        "frac_chg_abs", "frac_chg_p10", "chg_mean_p10", "chg_p90_p10",
        "frac_chg_p15", "chg_mean_p15", "chg_p90_p15", "massive_damage_guard",
        "spatial_entropy", "top10_share", "radial_decay", "com_shift", "axis_skew",
    )
    if int(vmask.sum()) < 4:
        for k in keys:
            out[k] = 0.0
        return out, np.ones(grid_sz * grid_sz, dtype=np.float32) / float(grid_sz * grid_sz)

    vals = cmap[vmask]
    p10 = float(np.percentile(vals, 10))
    p15 = float(np.percentile(vals, 15))
    crel10 = np.maximum(cmap - p10, 0.0) * vmask.astype(np.float32)
    crel15 = np.maximum(cmap - p15, 0.0) * vmask.astype(np.float32)
    vrel10 = crel10[vmask]
    vrel15 = crel15[vmask]

    out["frac_chg_abs"] = float(np.mean(vals > 0.25))
    out["frac_chg_p10"] = float(np.mean(vrel10 > 0.12))
    out["chg_mean_p10"] = float(vrel10.mean())
    out["chg_p90_p10"] = float(np.percentile(vrel10, 90))
    out["frac_chg_p15"] = float(np.mean(vrel15 > 0.12))
    out["chg_mean_p15"] = float(vrel15.mean())
    out["chg_p90_p15"] = float(np.percentile(vrel15, 90))
    out["massive_damage_guard"] = float((out["frac_chg_p15"] < 0.05) and (out["frac_chg_abs"] > 0.50))

    if out["massive_damage_guard"] > 0.5:
        crel_use = np.maximum(cmap - 0.10, 0.0) * vmask.astype(np.float32)
    else:
        crel_use = crel15
    vrel = crel_use[vmask]
    total = float(vrel.sum()) + 1e-8
    probs = vrel / total

    out["spatial_entropy"] = float(-np.sum(probs * np.log(probs + 1e-12)) / (np.log(len(probs) + 1e-12) + 1e-8))
    k10 = max(1, int(0.10 * len(vrel)))
    out["top10_share"] = float(np.sort(vrel)[-k10:].sum() / total)

    ys, xs = np.mgrid[0:grid_sz, 0:grid_sz].astype(np.float32)
    yv, xv = ys[vmask], xs[vmask]
    cy, cx = float(np.sum(probs * yv)), float(np.sum(probs * xv))
    out["com_shift"] = float(np.hypot(cy - (grid_sz - 1) / 2.0, cx - (grid_sz - 1) / 2.0) / (grid_sz * 0.5))

    pk = int(np.argmax(crel_use))
    py, px = float(pk // grid_sz), float(pk % grid_sz)
    dist = np.hypot(yv - py, xv - px)
    out["radial_decay"] = float(-np.corrcoef(dist, vrel)[0, 1]) if dist.std() > 1e-6 and vrel.std() > 1e-6 else 0.0

    coords = np.stack([xv - cx, yv - cy], axis=1)
    cov = (coords * probs[:, None]).T @ coords
    _, eigvecs = np.linalg.eigh(cov)
    proj = coords @ eigvecs[:, -1]
    sd_p = float(np.sqrt(np.sum(probs * (proj ** 2))) + 1e-6)
    out["axis_skew"] = float(abs(np.sum(probs * (proj ** 3)) / (sd_p ** 3)))

    weights = crel_use.reshape(-1)
    wsum = float(weights.sum())
    weights = (weights / wsum) if wsum > 1e-8 else (vmask.reshape(-1).astype(np.float32) / float(vmask.sum()))
    return out, weights.astype(np.float32)


class PairDataset(Dataset):
    def __init__(self, df: pd.DataFrame, img_size: int, mean, std, mode: str = "color_norm"):
        self.df = df.reset_index(drop=True)
        self.img_size = img_size
        self.mode = mode
        self.mean = np.array(mean, dtype=np.float32)[:, None, None]
        self.std = np.array(std, dtype=np.float32)[:, None, None]

    def __len__(self):
        return len(self.df)

    def _to_tensor_and_cloud(self, sq_img: np.ndarray, sq_msk: np.ndarray):
        view = make_gray3_clahe(sq_img) if self.mode == "gray3" else selective_color_norm(sq_img, sq_msk)
        arr = sq_img.astype(np.float32) / 255.0
        bright = arr.mean(axis=2)
        cmax, cmin = arr.max(axis=2), arr.min(axis=2)
        sat = (cmax - cmin) / (cmax + 1e-5)
        cld = ((bright > 0.85) & (sat < 0.14)).astype(np.float32)

        chw = ((view.transpose(2, 0, 1).astype(np.float32) / 255.0) - self.mean) / self.std
        chw[:, sq_msk <= 0.5] = 0.0
        return torch.from_numpy(chw), torch.from_numpy(sq_msk), torch.from_numpy(cld)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        bgr_p, bgr_q = cv2.imread(r["pre_path"]), cv2.imread(r["post_path"])
        if bgr_p is None or bgr_q is None:
            raise FileNotFoundError(f"Gagal membaca gambar: {r['Id']}")
        (sq_p, m_p), (sq_q, m_q), _ = crop_pad_inpaint_pair(
            cv2.cvtColor(bgr_p, cv2.COLOR_BGR2RGB),
            cv2.cvtColor(bgr_q, cv2.COLOR_BGR2RGB),
            self.img_size,
        )
        xp, mp, _ = self._to_tensor_and_cloud(sq_p, m_p)
        xq, mq, cq = self._to_tensor_and_cloud(sq_q, m_q)
        return xp, mp, xq, mq, cq


def select_cloud_excluded_mask(mq_grid: torch.Tensor, cq_grid: torch.Tensor):
    """Pool post/changed di atas patch bebas awan (<= CLOUD_PATCH_THR); fallback all_valid hanya jika <4 patch bersih."""
    vq_all = mq_grid >= 0.5
    vq_clean = vq_all & (cq_grid <= CLOUD_PATCH_THR)
    clean_cnt = vq_clean.reshape(vq_clean.shape[0], -1).sum(dim=1)
    fallback_flag = clean_cnt < 4
    vq_pool = torch.where(fallback_flag[:, None, None], vq_all, vq_clean)
    return vq_pool, vq_all, fallback_flag


def run_pairwise_extraction(
    model,
    df: pd.DataFrame,
    b_info: dict,
    mode: str = "color_norm",
    compute_patch_stats: bool = True,
    device: str = None,
    batch_size: int = BATCH_SIZE,
    num_workers: int = NUM_WORKERS,
    use_fp16: bool = USE_FP16,
):
    if device is None:
        device = b_info.get("device", "cuda" if torch.cuda.is_available() else "cpu")
    img_sz, p_sz, emb_dim = b_info["img_size"], b_info["patch_size"], b_info["dim"]
    g_sz = img_sz // p_sz
    ds = PairDataset(df, img_sz, b_info["mean"], b_info["std"], mode=mode)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=(device == "cuda"))

    N = len(df)
    ids = df["Id"].astype(str).tolist()
    emb_pre = np.zeros((N, emb_dim), dtype=np.float32)
    emb_post = np.zeros((N, emb_dim), dtype=np.float32)
    emb_chg = np.zeros((N, emb_dim), dtype=np.float32) if compute_patch_stats else None
    pstats = []
    ptr = 0

    with torch.no_grad():
        for xp, mp, xq, mq, cq in tqdm(dl, desc=f"Extract [{mode}]"):
            bs = xp.shape[0]
            xp, mp = xp.to(device, non_blocking=True), mp.to(device, non_blocking=True)
            xq, mq, cq = xq.to(device, non_blocking=True), mq.to(device, non_blocking=True), cq.to(device, non_blocking=True)

            vp_base = F.avg_pool2d(mp.unsqueeze(1), p_sz, p_sz).squeeze(1) >= 0.5
            mq_grid = F.avg_pool2d(mq.unsqueeze(1), p_sz, p_sz).squeeze(1)
            cq_grid = F.avg_pool2d(cq.unsqueeze(1), p_sz, p_sz).squeeze(1)
            vq_pool, vq_all, fb_flag = select_cloud_excluded_mask(mq_grid, cq_grid)

            acc_pre = torch.zeros((bs, emb_dim), device=device, dtype=torch.float32)
            acc_post = torch.zeros((bs, emb_dim), device=device, dtype=torch.float32)
            pre_bank = []
            post_tok0 = None

            for fl in (False, True):
                for rot in (0, 1, 2, 3):
                    xp_v, mp_v = d4_augment_batch(xp, vp_base, fl, rot)
                    xq_v, mq_v = d4_augment_batch(xq, vq_pool, fl, rot)

                    tp_n, mp_w, gp = extract_patch_grid(model, xp_v, mp_v, device=device, use_fp16=use_fp16)
                    tq_n, _, gq = extract_patch_grid(model, xq_v, mq_v, device=device, use_fp16=use_fp16)
                    acc_pre += gp
                    acc_post += gq

                    if compute_patch_stats:
                        pre_bank.append(tp_n * mp_w)
                        if not fl and rot == 0:
                            post_tok0 = tq_n

            emb_pre[ptr:ptr + bs] = F.normalize(acc_pre, dim=1).cpu().numpy()
            emb_post[ptr:ptr + bs] = F.normalize(acc_post, dim=1).cpu().numpy()

            if compute_patch_stats:
                bank_all = torch.cat(pre_bank, dim=1)
                sim = torch.bmm(post_tok0, bank_all.transpose(1, 2))
                max_sim, _ = sim.max(dim=2)
                chg_map = (1.0 - max_sim).reshape(bs, g_sz, g_sz).cpu().numpy()
                vp_np = vq_pool.cpu().numpy()
                va_np = vq_all.cpu().numpy()
                fb_np = fb_flag.cpu().numpy()
                pt_np = post_tok0.cpu().numpy()
                for i in range(bs):
                    st, w_chg = compute_spatial_stats(chg_map[i], vp_np[i], g_sz)
                    st_all, _ = compute_spatial_stats(chg_map[i], va_np[i], g_sz)
                    st["cloud_excluded_fallback_flag"] = float(fb_np[i])
                    for k_a, v_a in st_all.items():
                        st[f"{k_a}_allvalid"] = v_a
                    st["Id"] = ids[ptr + i]
                    pstats.append(st)
                    vec = (pt_np[i] * w_chg[:, None]).sum(axis=0)
                    emb_chg[ptr + i] = vec / (np.linalg.norm(vec) + 1e-8)

            ptr += bs

    return emb_pre, emb_post, emb_chg, pd.DataFrame(pstats) if compute_patch_stats else None


def compute_tes_a_retrieval(pre_emb: np.ndarray, post_emb: np.ndarray, ks=(1, 5, 10)) -> dict:
    """Tes A: pencocokan post -> pre per pasangan (tanpa label), mengukur ketahanan preprocessing."""
    n = pre_emb.shape[0]
    p_norm = pre_emb / (np.linalg.norm(pre_emb, axis=1, keepdims=True) + 1e-8)
    q_norm = post_emb / (np.linalg.norm(post_emb, axis=1, keepdims=True) + 1e-8)
    sim = q_norm @ p_norm.T
    ranks = np.argsort(-sim, axis=1)
    targets = np.arange(n)[:, None]
    out = {}
    for k in ks:
        k_use = min(int(k), n)
        out[f"Recall@{k}"] = float(np.mean(np.any(ranks[:, :k_use] == targets, axis=1)))
    return out


def compute_artifact_and_phys(rgb_raw: np.ndarray) -> dict:
    """Ekstraksi fitur artifak & sinyal fisik bencana per citra (skala [0, 1] untuk pink_cast)."""
    small = cv2.resize(rgb_raw, (256, 256), interpolation=cv2.INTER_AREA)
    valid = detect_valid_mask(small)
    if int(valid.sum()) < 50:
        valid = np.ones((256, 256), dtype=bool)
    arr = small.astype(np.float32) / 255.0
    rv, gv, bv = arr[:, :, 0][valid], arr[:, :, 1][valid], arr[:, :, 2][valid]
    bright = (rv + gv + bv) / 3.0
    cmax, cmin = np.maximum(np.maximum(rv, gv), bv), np.minimum(np.minimum(rv, gv), bv)
    sat = (cmax - cmin) / (cmax + 1e-5)

    pink_cast = float(max(0.0, min(float(rv.mean() - gv.mean()), float(bv.mean() - gv.mean()))))
    chroma_std = float(np.mean(np.std(arr[valid], axis=1)))
    gray_u8 = cv2.cvtColor(small, cv2.COLOR_RGB2GRAY)
    gy = np.abs(cv2.Sobel(gray_u8, cv2.CV_32F, 0, 1))
    gx = np.abs(cv2.Sobel(gray_u8, cv2.CV_32F, 1, 0))
    lap = float(cv2.Laplacian(gray_u8, cv2.CV_32F)[valid].var())

    return {
        "black_frac": float(1.0 - valid.mean()),
        "cloud_frac": float(np.mean((bright > 0.85) & (sat < 0.14))),
        "tint_score": pink_cast,
        "is_gray": float(chroma_std < 0.02),
        "chroma_std": chroma_std,
        "brightness": float(bright.mean()),
        "sat": float(sat.mean()),
        "blur_score": lap,
        "scanline_score": float((gy[valid].mean() + 1e-5) / (gx[valid].mean() + 1e-5)),
        "exg": float(np.mean(2.0 * gv - rv - bv)),
        "water": float(np.mean((bv > rv + 0.03) & (bv > gv + 0.01) & (bright < 0.65))),
        "dark_char": float(np.mean(bright < 0.22)),
        "ash_ratio": float(np.mean((sat < 0.14) & (bright >= 0.25) & (bright <= 0.72))),
        "edge": float(np.hypot(gx, gy)[valid].mean() / 255.0),
    }

In [3]:
set_reproducible_seed(SEED)
HF_TOKEN = get_hf_token()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WORK_ROOT = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
OUT_DIR = os.path.join(WORK_ROOT, "v4_dino_features")
WEIGHTS_DIR = os.path.join(WORK_ROOT, "weights")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(WEIGHTS_DIR, exist_ok=True)

print(f"torch={torch.__version__} | transformers={getattr(transformers, '__version__', 'N/A')} | timm={getattr(timm, '__version__', 'N/A')}")
print("[OK] HF_TOKEN terdeteksi." if HF_TOKEN else "[INFO] HF_TOKEN tidak ditemukan; coba offline weights/ dulu lalu fallback jika gated.")


def find_task2_paths():
    for sr in [
        "/kaggle/input/competitions/task-2-datavidia-11-0",
        "/kaggle/input/task-2-datavidia-11-0",
        "/kaggle/input",
        ".",
    ]:
        if not os.path.exists(sr):
            continue
        for root, _, files in os.walk(sr):
            if "sample_submission.csv" in files:
                cand_sub = os.path.join(root, "sample_submission.csv")
                probe_id = str(pd.read_csv(cand_sub, nrows=1)["Id"].iloc[0])
                if os.path.exists(os.path.join(root, probe_id, "pre.jpg")):
                    return root, cand_sub
                if os.path.exists(os.path.join(root, "Fenomena", probe_id, "pre.jpg")):
                    return os.path.join(root, "Fenomena"), cand_sub
    raise FileNotFoundError("Dataset Task-2 (sample_submission.csv + <Id>/pre.jpg) tidak ditemukan.")


DATA_DIR, SUB_PATH = find_task2_paths()
sub_df = pd.read_csv(SUB_PATH)
all_ids = sub_df["Id"].astype(str).tolist()
pairs_df = pd.DataFrame({
    "Id": all_ids,
    "pre_path": [os.path.join(DATA_DIR, uid, "pre.jpg") for uid in all_ids],
    "post_path": [os.path.join(DATA_DIR, uid, "post.jpg") for uid in all_ids],
})
assert all(os.path.exists(p) for p in pairs_df["pre_path"].head(5)), "Path pre.jpg contoh tidak ditemukan!"
print(f"DEVICE={DEVICE} (GPUs={torch.cuda.device_count()}) | DATA_DIR={DATA_DIR} | OUT_DIR={OUT_DIR} | WEIGHTS_DIR={WEIGHTS_DIR} | N={len(pairs_df)}")

torch=2.11.0+cu130 | transformers=5.24.1 | timm=1.0.30
[OK] HF_TOKEN terdeteksi.
DEVICE=cuda (GPUs=2) | DATA_DIR=/kaggle/input/competitions/task-2-datavidia-11-0/Task-2/Fenomena | OUT_DIR=/kaggle/working/v4_dino_features | WEIGHTS_DIR=/kaggle/working/weights | N=3180


In [4]:
t_start = time.time()
print("[TAHAP 1/4] Menghitung Artifak & Fitur Fisik dasar...")
art_rows = []
for _, row in tqdm(pairs_df.iterrows(), total=len(pairs_df), desc="Artifak & Fisik"):
    mp = compute_artifact_and_phys(cv2.cvtColor(cv2.imread(row["pre_path"]), cv2.COLOR_BGR2RGB))
    mq = compute_artifact_and_phys(cv2.cvtColor(cv2.imread(row["post_path"]), cv2.COLOR_BGR2RGB))
    r = {"Id": row["Id"], **{f"{k}_pre": v for k, v in mp.items()}, **{f"{k}_post": v for k, v in mq.items()}}
    unrel = (mp["is_gray"] > 0.5) or (mq["is_gray"] > 0.5) or (mp["tint_score"] > TINT_NORM_THR) or (mq["tint_score"] > TINT_NORM_THR)
    r["color_unreliable_flag"] = float(unrel)
    r["delta_exg"] = mq["exg"] - mp["exg"]
    r["delta_water"] = mq["water"] - mp["water"]
    r["ash_gain"] = mq["ash_ratio"] - mp["ash_ratio"]
    r["sat_drop"] = mp["sat"] - mq["sat"]
    r["dark_char_gain"] = mq["dark_char"] - mp["dark_char"]
    r["delta_edge"] = mq["edge"] - mp["edge"]
    r["edge_ratio"] = (mq["edge"] + 1e-4) / (mp["edge"] + 1e-4)
    for ccol in ("delta_exg", "delta_water", "ash_gain", "sat_drop", "dark_char_gain"):
        r[f"{ccol}_nan"] = np.nan if unrel else r[ccol]
        r[f"{ccol}_imp0"] = 0.0 if unrel else r[ccol]
    art_rows.append(r)

art_df = pd.DataFrame(art_rows)
art_df.to_parquet(os.path.join(OUT_DIR, "artifact_and_phys_v3.parquet"), index=False)
print(f"[TAHAP 1 SELESAI] artifact_and_phys_v3.parquet tersimpan ({art_df.shape}). Memuat backbone...")

backbone, b_info = load_dino_backbone(
    backbone_choice=BACKBONE_CHOICE,
    device=DEVICE,
    hf_token=HF_TOKEN,
    save_weights_dir=WEIGHTS_DIR,
)
if ALLOW_DATAPARALLEL and DEVICE == "cuda" and torch.cuda.device_count() > 1:
    backbone = nn.DataParallel(backbone)
print("BACKBONE:", json.dumps(b_info, indent=2))

print("[TAHAP 2/4] Ekstraksi GPU cabang warna [color_norm] + patch change stats...")
c_pre, c_post, c_chg, patch_df = run_pairwise_extraction(backbone, pairs_df, b_info, mode="color_norm", compute_patch_stats=True, device=DEVICE)
np.save(os.path.join(OUT_DIR, "dino_color_pre.npy"), c_pre)
np.save(os.path.join(OUT_DIR, "dino_color_post.npy"), c_post)
np.save(os.path.join(OUT_DIR, "dino_changed_embed.npy"), c_chg)
patch_df.to_parquet(os.path.join(OUT_DIR, "patch_change_stats_v3.parquet"), index=False)
print(f"[TAHAP 2 SELESAI] color_norm & patch_change_stats_v3.parquet ({patch_df.shape}) tersimpan.")

tes_a_color = compute_tes_a_retrieval(c_pre, c_post)
tes_a_gray = None

if EXTRACT_GRAY3:
    print("[TAHAP 3/4] Ekstraksi GPU cabang grayscale CLAHE [gray3]...")
    g_pre, g_post, _, _ = run_pairwise_extraction(backbone, pairs_df, b_info, mode="gray3", compute_patch_stats=False, device=DEVICE)
    np.save(os.path.join(OUT_DIR, "dino_gray_pre.npy"), g_pre)
    np.save(os.path.join(OUT_DIR, "dino_gray_post.npy"), g_post)
    tes_a_gray = compute_tes_a_retrieval(g_pre, g_post)
    print("[TAHAP 3 SELESAI] dino_gray_pre/post.npy tersimpan.")

b_info["tes_a_color"] = tes_a_color
if tes_a_gray is not None:
    b_info["tes_a_gray"] = tes_a_gray
b_info["extracted_at"] = time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime())

for meta_name in ("extract_v4_meta.json", "meta.json"):
    with open(os.path.join(OUT_DIR, meta_name), "w", encoding="utf-8") as f:
        json.dump(b_info, f, indent=2)

reqs = [
    f"torch=={torch.__version__}",
    f"transformers=={getattr(transformers, '__version__', 'N/A')}",
    f"timm=={getattr(timm, '__version__', 'N/A')}",
    f"numpy=={np.__version__}",
    f"pandas=={pd.__version__}",
    f"opencv-python=={cv2.__version__}",
]
with open(os.path.join(OUT_DIR, "requirements_extract.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(reqs) + "\n")

assert c_pre.shape == (len(pairs_df), b_info["dim"]) and not np.isnan(c_pre).any()
assert c_post.shape == (len(pairs_df), b_info["dim"]) and not np.isnan(c_post).any()
assert c_chg.shape == (len(pairs_df), b_info["dim"]) and not np.isnan(c_chg).any()
print(f"TES A [color]: {tes_a_color}")
if tes_a_gray is not None:
    print(f"TES A [gray3]: {tes_a_gray}")
print(f"SELESAI GPU dalam {time.time() - t_start:.1f}s! Artefak di: {OUT_DIR} | Bobot di: {WEIGHTS_DIR}")
print(f"Tint flagged pairs (legacy): {int(art_df['color_unreliable_flag'].sum())}/{len(art_df)}")

[TAHAP 1/4] Menghitung Artifak & Fitur Fisik dasar...


Artifak & Fisik:   0%|          | 0/3180 [00:00<?, ?it/s]

[TAHAP 1 SELESAI] artifact_and_phys_v3.parquet tersimpan ((3180, 47)). Memuat backbone...


preprocessor_config.json:   0%|          | 0.00/585 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/745 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.21GB            

... (4 outputs left out)

Extract [color_norm]:   0%|          | 0/199 [00:00<?, ?it/s]

[TAHAP 2 SELESAI] color_norm & patch_change_stats_v3.parquet ((3180, 28)) tersimpan.
[TAHAP 3/4] Ekstraksi GPU cabang grayscale CLAHE [gray3]...


Extract [gray3]:   0%|          | 0/199 [00:00<?, ?it/s]

[TAHAP 3 SELESAI] dino_gray_pre/post.npy tersimpan.
TES A [color]: {'Recall@1': 0.06823899371069182, 'Recall@5': 0.1279874213836478, 'Recall@10': 0.16761006289308175}
TES A [gray3]: {'Recall@1': 0.061635220125786164, 'Recall@5': 0.11981132075471698, 'Recall@10': 0.15377358490566037}
SELESAI GPU dalam 2102.0s! Artefak di: /kaggle/working/v4_dino_features | Bobot di: /kaggle/working/weights
Tint flagged pairs (legacy): 922/3180


In [5]:
try:
    from skimage.filters.rank import entropy as _rank_entropy
    from skimage.morphology import disk as _disk
    HAS_SKIMAGE = True
except ImportError:
    HAS_SKIMAGE = False


# ==================== KONFIGURASI PHYS V4 ====================
AMBANG_COLOR_SHIFT = 15.0        # Skala 0-255 sqrt((rg_post-rg_pre)^2 + (bg_post-bg_pre)^2)
AMBANG_HAZE_STD = 25.0           # std(gray) < 25
AMBANG_HAZE_MEAN = 150.0         # mean(gray) > 150
BIRU_REL_BG_THR = 12.0           # (B-G) - median(B-G) > 12 (skala 0-255)
BIRU_REL_BR_THR = 12.0           # (B-R) - median(B-R) > 12 (skala 0-255)
WATER_TEX_THR = 18.0             # Tekstur gradient rendah untuk piksel air relatif
# ============================================================


def _local_entropy_mean(gray_u8: np.ndarray, valid: np.ndarray) -> float:
    """Hitung rata-rata entropi lokal pada grayscale CLAHE (hanya di piksel valid)."""
    clahe_g = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray_u8)
    if HAS_SKIMAGE:
        ent = _rank_entropy(clahe_g, _disk(5)).astype(np.float32)
        return float(ent[valid].mean())
    # Fallback cepat berbasis varians lokal ternormalisasi log bila scikit-image tidak tersedia
    blur1 = cv2.GaussianBlur(clahe_g.astype(np.float32), (11, 11), 0)
    blur2 = cv2.GaussianBlur((clahe_g.astype(np.float32) ** 2), (11, 11), 0)
    lstd = np.sqrt(np.maximum(blur2 - blur1 ** 2, 1.0))
    return float(np.log2(lstd[valid] + 1.0).mean())


def _extract_single_phys_v4(rgb_raw: np.ndarray, tint_thr: float = TINT_NORM_THR) -> dict:
    small = cv2.resize(rgb_raw, (256, 256), interpolation=cv2.INTER_AREA)
    valid = detect_valid_mask(small)
    if int(valid.sum()) < 50:
        valid = np.ones((256, 256), dtype=bool)

    arr255 = small.astype(np.float32)
    arr01 = arr255 / 255.0
    rv255, gv255, bv255 = arr255[:, :, 0][valid], arr255[:, :, 1][valid], arr255[:, :, 2][valid]
    rv, gv, bv = arr01[:, :, 0][valid], arr01[:, :, 1][valid], arr01[:, :, 2][valid]

    gray_u8 = cv2.cvtColor(small, cv2.COLOR_RGB2GRAY)
    gv_u8 = gray_u8[valid].astype(np.float32)
    med_gray = float(np.median(gv_u8))
    dark_rel = float(np.mean(gv_u8 < (0.5 * med_gray)))

    rg = float(np.mean(rv255 - gv255))
    bg = float(np.mean(bv255 - gv255))
    tint_pink = float(max(0.0, min(float(rv.mean() - gv.mean()), float(bv.mean() - gv.mean()))))
    chroma_std = float(np.mean(np.std(arr01[valid], axis=1)))
    is_gray = float(chroma_std < 0.02)
    haze_flag = float((float(gv_u8.std()) < AMBANG_HAZE_STD) and (float(gv_u8.mean()) > AMBANG_HAZE_MEAN))

    # Air relatif terhadap median gambar sendiri + tekstur gradient rendah
    bg_diff = arr255[:, :, 2] - arr255[:, :, 1]
    br_diff = arr255[:, :, 2] - arr255[:, :, 0]
    med_bg = float(np.median(bg_diff[valid]))
    med_br = float(np.median(br_diff[valid]))
    gx = np.abs(cv2.Sobel(gray_u8, cv2.CV_32F, 1, 0))
    gy = np.abs(cv2.Sobel(gray_u8, cv2.CV_32F, 0, 1))
    grad = np.hypot(gx, gy)
    bright_px = arr01.mean(axis=2)
    water_mask = (
        ((bg_diff - med_bg) > BIRU_REL_BG_THR)
        & ((br_diff - med_br) > BIRU_REL_BR_THR)
        & (grad < WATER_TEX_THR)
        & (bright_px < 0.70)
    )
    water_rel = float(water_mask[valid].mean())

    cmax, cmin = np.maximum(np.maximum(rv, gv), bv), np.minimum(np.minimum(rv, gv), bv)
    sat = (cmax - cmin) / (cmax + 1e-5)
    bright_v = (rv + gv + bv) / 3.0

    return {
        "rg": rg,
        "bg": bg,
        "tint_pink": float(tint_pink > tint_thr),
        "is_gray": is_gray,
        "haze_flag": haze_flag,
        "dark_rel": dark_rel,
        "water_rel": water_rel,
        "entropy": _local_entropy_mean(gray_u8, valid),
        "exg": float(np.mean(2.0 * gv - rv - bv)),
        "ash_ratio": float(np.mean((sat < 0.14) & (bright_v >= 0.25) & (bright_v <= 0.72))),
        "sat": float(sat.mean()),
        "brightness": float(bright_v.mean()),
    }


def compute_phys_v4_pair(uid: str, rgb_pre: np.ndarray, rgb_post: np.ndarray) -> dict:
    """Hitung 23 kolom phys_v4.parquet sesuai perubahan_preprocessing_v4.md."""
    sp = _extract_single_phys_v4(rgb_pre)
    sq = _extract_single_phys_v4(rgb_post)

    color_shift = float(np.hypot(sq["rg"] - sp["rg"], sq["bg"] - sp["bg"]))
    color_shift_flag = float(color_shift > AMBANG_COLOR_SHIFT)
    unrel = bool(
        (sp["is_gray"] > 0.5)
        or (sq["is_gray"] > 0.5)
        or (sp["tint_pink"] > 0.5)
        or (sq["tint_pink"] > 0.5)
        or (color_shift_flag > 0.5)
        or (sp["haze_flag"] > 0.5)
        or (sq["haze_flag"] > 0.5)
    )

    water_rel_pre = np.nan if unrel else sp["water_rel"]
    water_rel_post = np.nan if unrel else sq["water_rel"]
    d_water = np.nan if unrel else (sq["water_rel"] - sp["water_rel"])

    return {
        "Id": uid,
        "rg_pre": sp["rg"],
        "bg_pre": sp["bg"],
        "rg_post": sq["rg"],
        "bg_post": sq["bg"],
        "color_shift": color_shift,
        "color_shift_flag": color_shift_flag,
        "haze_flag_pre": sp["haze_flag"],
        "haze_flag_post": sq["haze_flag"],
        "color_unreliable_v4": float(unrel),
        "dark_rel_pre": sp["dark_rel"],
        "dark_rel_post": sq["dark_rel"],
        "d_dark": sq["dark_rel"] - sp["dark_rel"],
        "water_rel_pre": water_rel_pre,
        "water_rel_post": water_rel_post,
        "d_water": d_water,
        "entropy_pre": sp["entropy"],
        "entropy_post": sq["entropy"],
        "d_entropy": sq["entropy"] - sp["entropy"],
        "d_exg_v4": np.nan if unrel else (sq["exg"] - sp["exg"]),
        "ash_gain_v4": np.nan if unrel else (sq["ash_ratio"] - sp["ash_ratio"]),
        "sat_drop_v4": np.nan if unrel else (sp["sat"] - sq["sat"]),
        "d_bright_raw_v4": np.nan if unrel else (sq["brightness"] - sp["brightness"]),
        "_raw_d_bright": sq["brightness"] - sp["brightness"],
    }


def audit_phys_v4_dataframe(phys_df: pd.DataFrame, ref_ids: list):
    """Jalankan seluruh pengecekan Bagian 5 perubahan_preprocessing_v4.md."""
    assert list(phys_df["Id"]) == ref_ids, "Urutan Id phys_v4.parquet tidak cocok dengan sample_submission.csv!"
    raw_db = phys_df["_raw_d_bright"].values
    d_dark = phys_df["d_dark"].values
    corr_dark = float(np.corrcoef(d_dark, raw_db)[0, 1]) if raw_db.std() > 1e-8 and d_dark.std() > 1e-8 else 0.0
    unrel_frac = float(phys_df["color_unreliable_v4"].mean())
    print("=== AUDIT PHYS_V4 (BAGIAN 5) ===")
    print(f"Rows: {len(phys_df)} | |corr(d_dark, d_brightness_raw)| = {abs(corr_dark):.4f} (Target < 0.30)")
    print(f"color_unreliable_v4: {int(phys_df['color_unreliable_v4'].sum())}/{len(phys_df)} ({unrel_frac*100:.1f}%)")
    if unrel_frac > 0.50:
        print("[PERINGATAN] Porsi color_unreliable_v4 > 50% — pertimbangkan melonggarkan AMBANG_COLOR_SHIFT / AMBANG_HAZE.")
    for probe_id in ("d62c43e0f31c", "63431151b5b5"):
        sub = phys_df[phys_df["Id"] == probe_id]
        if len(sub) > 0:
            r = sub.iloc[0]
            print(f"  Probe {probe_id}: color_unreliable_v4={r['color_unreliable_v4']} | color_shift={r['color_shift']:.1f} | "
                  f"haze=({r['haze_flag_pre']},{r['haze_flag_post']}) | water_rel_pre={r['water_rel_pre']}")
    return corr_dark, unrel_frac

In [6]:
print("[TAHAP 4/4] Menghitung phys_v4.parquet + Audit Bagian 5...")
phys_v4_rows = []
for _, row in tqdm(pairs_df.iterrows(), total=len(pairs_df), desc="Hitung phys_v4"):
    rgb_p = cv2.cvtColor(cv2.imread(row["pre_path"]), cv2.COLOR_BGR2RGB)
    rgb_q = cv2.cvtColor(cv2.imread(row["post_path"]), cv2.COLOR_BGR2RGB)
    phys_v4_rows.append(compute_phys_v4_pair(row["Id"], rgb_p, rgb_q))

phys_v4_full = pd.DataFrame(phys_v4_rows)
audit_phys_v4_dataframe(phys_v4_full, all_ids)
phys_v4_cols = [c for c in phys_v4_full.columns if c != "_raw_d_bright"]
phys_v4_out = os.path.join(OUT_DIR, "phys_v4.parquet")
phys_v4_full[phys_v4_cols].to_parquet(phys_v4_out, index=False)
print(f"[OK] Tersimpan file baru: {phys_v4_out} ({phys_v4_full[phys_v4_cols].shape})")

[TAHAP 4/4] Menghitung phys_v4.parquet + Audit Bagian 5...


Hitung phys_v4:   0%|          | 0/3180 [00:00<?, ?it/s]

=== AUDIT PHYS_V4 (BAGIAN 5) ===
Rows: 3180 | |corr(d_dark, d_brightness_raw)| = 0.4284 (Target < 0.30)
color_unreliable_v4: 2583/3180 (81.2%)
[PERINGATAN] Porsi color_unreliable_v4 > 50% — pertimbangkan melonggarkan AMBANG_COLOR_SHIFT / AMBANG_HAZE.
  Probe d62c43e0f31c: color_unreliable_v4=1.0 | color_shift=26.2 | haze=(0.0,0.0) | water_rel_pre=nan
  Probe 63431151b5b5: color_unreliable_v4=1.0 | color_shift=29.8 | haze=(0.0,1.0) | water_rel_pre=nan
[OK] Tersimpan file baru: /kaggle/working/v4_dino_features/phys_v4.parquet ((3180, 23))
